<a href="https://colab.research.google.com/github/pia-francesca/ema/blob/main/examples/Pla2g2/emmaemb_pla2g2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EmmaEmb: Comparative Analysis of Embedding Spaces

Welcome to the example Colab notebook for EmmaEmb, a Python library for analyzing and comparing embedding spaces in molecular biology. EmmaEmb provides tools to explore how different embedding models capture biological information, enabling insights into feature similarities, differences, and relationships across embeddings.

Link to GitHub: https://github.com/broadinstitute/EmmaEmb


### Notebook content

This notebook demonstrates key functionalities of EmmaEmb, including:

1. [Initialising the Emma object](#section-one)

2. [Adding embedding spaces](#section-two)

3. [Embedding space diagnostics](#section-three)

4. [Feature distribution across spaces](#section-four)

5. [Pairwise space comparison](#section-five)

![EmmaEmb Overview](https://raw.githubusercontent.com/broadinstitute/EmmaEmb/main/images/emma_overview.jpg)


## 0. Loading dependencies and data

Information of the data and embedding models can be found here: https://github.com/broadinstitute/EmmaEmb/tree/main/examples/Pla2g2

In [ ]:
#@title Install dependencies
%pip install emmaemb

In [ ]:
#@title Download example data from EmmaEmb repository

import requests
import pandas as pd
import os

# download embeddings

models = ["ESMC", "ProtT5"]
embedding_url_dir = "https://raw.githubusercontent.com/broadinstitute/EmmaEmb/main/examples/Pla2g2/embeddings/"

headers = {"User-Agent": "Mozilla/5.0"}
csv_url = "https://raw.githubusercontent.com/broadinstitute/EmmaEmb/main/examples/Pla2g2/Pla2g2_features.csv"

csv_filename = "Pla2g2_features.csv"
csv_response = requests.get(csv_url, headers=headers)
if csv_response.status_code == 200:
    with open(csv_filename, "wb") as f:
        f.write(csv_response.content)
else:
    print(f"Failed to download {csv_filename}")


df_pla2g2 = pd.read_csv(csv_filename)
proteins = df_pla2g2['identifier'].values

# now for each model download embedding files for each protein
for model in models:
  model_dir = f"embeddings/{model}"
  os.makedirs(model_dir, exist_ok=True)

  for protein in proteins:
    file_path = os.path.join(model_dir, f"{protein}.npy")

    # Check if file already exists
    if os.path.exists(file_path):
        continue

    url = f"{embedding_url_dir}{model}/{protein}.npy"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
      # store in embeddings/model/protein-id.npy
      with open("embeddings/" + model + "/" + protein + ".npy", "wb") as f:
        f.write(response.content)
    else:
      print(f"Failed to download {url}")


print("All download of feature data complete.")

<a name="section-one"></a>
## 1. Initialising the Emma object

In [ ]:
import pandas as pd
from emmaemb import Emma

### Feature data table

Loading feature data. The first column includes the identifiers of the samples, in this case proteins. The remaining columns contain meta data on each sample.

In [ ]:
df_pla2f2 = pd.read_csv("Pla2g2_features.csv")
print(df_pla2f2.shape)
df_pla2f2.head()

Initialising Emma object with the feature data in format of a pandas df. The datatype stored in each column of the feature data is detected. Only cateorical data will be available for downstream analysis with the Emma library.

Note: Quantitative features can be binned to allow analysis with EmmaEmb.

In [ ]:
# initiate Emma object with the metadata

emma = Emma(df_pla2f2)

<a name="section-two"></a>
## 2. Adding embedding spaces

Adding embedding spaces. Embedding spaces can be added one by one. Either by

- providing a link to a directory which stores the embeddings in individual files with the identifiers from the feature table or

- by providing a numpy array which includes the embeddings in each row and in the same order as in the feature data table.


Multiple embedding spaces can be added. Dimensions of the embeddings do not have to be the same across embedding spaces. Embedding spaces can be removed using the `remove_emb_space(emb_space_name: str)` function.

In [ ]:
embedding_dir = "embeddings/"
models = ["ProtT5", "ESMC"]

In [ ]:
for model_name in models:
    emma.add_emb_space(
        embeddings_source=embedding_dir + model_name,
        emb_space_name=model_name,
    )

### Visualization of embedding spaces using dimensionality reduction techniques

The `plot_emb_space` function visualizes the embeddings of a specified embedding space in 2D using dimensionality reduction techniques such as PCA, t-SNE, or UMAP. It takes an Emma instance containing multiple embedding spaces and projects the selected space into two dimensions, optionally normalizing the data beforehand. The resulting scatter plot can be colored based on metadata attributes, allowing for an intuitive exploration of patterns within the embedding space.

In [ ]:
# visualise reduced embedding space
from emmaemb.visualization import plot_emb_space

fig_pca = plot_emb_space(emma=emma,
                         emb_space="ProtT5",
                         method="PCA",
                         color_by="enzyme_class",
                         normalize=True)
fig_pca.show()

<a name="section-three"></a>
## 3. Embedding space diagnostics

Before computing pairwise distances and alignment scores, it is useful to assess the geometric quality of each embedding space. EmmaEmb provides two diagnostics: **anisotropy** (how uniformly the space is occupied) and **hubness** (whether a few points dominate nearest-neighbor lists). These inform the choice of distance metric and the interpretation of downstream scores.

### 3.1 Anisotropy

An isotropic embedding space is one where vectors point in many different directions. High anisotropy — where most embeddings cluster along a narrow cone — can inflate cosine similarities and distort nearest-neighbor structure.

`get_anisotropy_diagnostics` measures the average pairwise cosine similarity between random vector pairs. Values close to 0 indicate near-isotropic geometry; values approaching 1 indicate severe anisotropy. The diagnostic also reports whether mean-centering is likely to help.

In [ ]:
from emmaemb.functions import get_anisotropy_diagnostics

aniso = get_anisotropy_diagnostics(emma, n_pairs=10_000, seed=42)
print(aniso)

If anisotropy is detected, mean-centering can reduce it. `emma.mean_center()` subtracts the per-dimension mean in-place. The original embeddings are preserved and can be restored at any time with `emma.revert_mean_centering()`.

In [ ]:
# Uncomment to apply mean-centering if anisotropy is high
emma.mean_center()

# Any cached pairwise distances are cleared automatically — recompute after centering:
emma.calculate_pairwise_distances("ProtT5", "cityblock")
emma.calculate_pairwise_distances("ESMC", "cityblock")

### 3.2 Hubness

Hubness is a phenomenon in high-dimensional spaces where a small number of points become the nearest neighbors of a disproportionately large number of others — not because they are genuinely similar to everything, but as a mathematical consequence of high dimensionality. Hub points can inflate KNN-based alignment scores and bias downstream analyses.

`get_hubness_diagnostics` returns the Robin Hood index (a summary of how unequally k-occurrences are distributed across samples) and the per-sample k-occurrence distribution.

In [ ]:
from emmaemb.functions import get_hubness_diagnostics

hub = get_hubness_diagnostics(emma, k=10, metric="cosine")
print(hub)

<a name="section-four"></a>
## 4. Feature distribution across spaces

The `calculate_pairwise_distances` method computes pairwise distances between samples in an embedding space and caches the k-nearest neighbor ranks for downstream analyses. Supported distance metrics include Euclidean, Manhattan, Cosine, and several normalized variants. Distances are only computed once — subsequent calls for the same space and metric return immediately.

In [ ]:
pwd = emma.calculate_pairwise_distances("ProtT5", "cityblock")
pwd = emma.calculate_pairwise_distances("ESMC", "cityblock")

#### 4.1 KNN feature alignment scores

### 4.3 KNN alignment across k

The choice of k affects alignment scores. `plot_knn_alignment_across_k` sweeps k across a range and plots the mean alignment score for each embedding space. Stable rankings across k indicate robust results; crossings or inversions suggest parameter sensitivity and should be reported.

In [ ]:
from emmaemb.visualization import plot_knn_alignment_across_k

fig_knn_k = plot_knn_alignment_across_k(
    emma=emma,
    feature="enzyme_class",
    k_values=[5, 10, 20, 50, 100],
    metrics=["cityblock"],
    show_random_baselines=True,
    elbow_detection=True,
)
fig_knn_k.update_layout(height=500, width=700)
fig_knn_k.show()

### 4.4 Within/between class distance distributions

`plot_within_between_distributions` visualises the overlap between within-class and between-class pairwise distances for a given embedding space. A clean separation indicates that the embedding geometry reflects the class structure well; large overlap suggests poor class separation for the chosen metric.

In [ ]:
from emmaemb.visualization import plot_within_between_distributions

fig_wb = plot_within_between_distributions(
    emma=emma,
    emb_space="ProtT5",
    metric="cityblock",
    feature="enzyme_class",
)
fig_wb.update_layout(height=500, width=700)
fig_wb.show()

The `plot_knn_alignment_across_embedding_spaces` function visualizes k-nearest neighbor (KNN) feature alignment scores for a specified feature across multiple embedding spaces.
It computes what fraction of the KNN embeddings are labelled with the same label for the selected feature.
The scores are calculated for each embedding in each embedding space.
Pairwise distances need to be pre-computed for each embedding space and each distance metric by calling `calculate_pairwise_distances` beforehand (see above).
The function produces a box plot and allows customization of the embedding space order and plot color.

In [ ]:
# KNN ALIGNMENT SCORES
from emmaemb.visualization import plot_knn_alignment_across_embedding_spaces

fig_alignment_scores = plot_knn_alignment_across_embedding_spaces(
    emma, feature="enzyme_class", k=10, metric="cityblock"
)
fig_alignment_scores.update_layout(height=600, width=500)
fig_alignment_scores.show()

The KNN feature alignment scores can also be aggregated. The `plot_knn_alignment_across_classes` shows a heatmap of the mean value of the KNN feature alignment scores stratified by embedding space and feature class.

### 5.3 Robustness to class imbalance and label noise

Alignment scores can be affected by class imbalance (large classes dominate neighbors) and label quality. The following two plots assess robustness to these factors, helping distinguish genuine geometric class structure from frequency-driven or noise-sensitive artefacts.

#### 5.3a Class imbalance

`plot_knn_alignment_vs_class_balance` progressively downsamples the majority class toward the size of the smallest class and repeats KNN alignment. Stable rankings across this sweep indicate that the result is not driven by class frequency.

In [ ]:
from emmaemb.visualization import plot_knn_alignment_vs_class_balance

fig_balance = plot_knn_alignment_vs_class_balance(
    emma=emma,
    feature="enzyme_class",
    emb_spaces=["ProtT5", "ESMC"],
    k_values=[10, 50],
    metrics=["cityblock"],
    n_balance_steps=6,
    seed=42,
    show_random_baselines=True,
)
fig_balance.update_layout(height=500, width=700)
fig_balance.show()

#### 5.3b Label noise

`plot_knn_alignment_vs_feature_noise` progressively corrupts a fraction of labels while keeping the embedding geometry fixed. A score that degrades gracefully with noise indicates that the class structure is genuinely encoded in the embeddings, rather than being an artefact of a few perfectly labelled hubs.

In [ ]:
from emmaemb.visualization import plot_knn_alignment_vs_feature_noise

fig_noise = plot_knn_alignment_vs_feature_noise(
    emma=emma,
    feature="enzyme_class",
    emb_spaces=["ProtT5", "ESMC"],
    k_values=[10, 50],
    metrics=["cityblock"],
    n_noise_steps=8,
    n_repeats=3,
    seed=42,
    show_random_baselines=True,
)
fig_noise.update_layout(height=500, width=700)
fig_noise.show()

In [ ]:
from emmaemb.visualization import plot_knn_alignment_across_classes

fig_alignment_scores_class = plot_knn_alignment_across_classes(
    emma, feature="enzyme_class", k=100, metric="cityblock"
)
fig_alignment_scores_class.update_layout(height=600, width=500)
fig_alignment_scores_class.show()

### 4.2 KNN class mixing matrix

The KNN class mixing matrix quantifies the mixing of classes within the KNN neighborhood of samples in a given embedding space.
Given a distance metric (default: euclidean), the KNN are retrieved for each sample and the KNN class `get_class_mixing_in_neighborhood` counts how often different classes appear among its neighbors. The function returns a class mixing matrix, where each entry represents the number of times a class appears in the neighborhood of another class, along with the unique class labels. The heatmap can be visualised using the `plot_knn_class_mixing_matrix` function.

In [ ]:
# KNN CLASS MIXING MATRIX
from emmaemb.visualization import plot_knn_class_mixing_matrix

fig_class_mixing_matrix = plot_knn_class_mixing_matrix(
    emma,
    emb_space="ProtT5",
    feature="enzyme_class",
    k=100,
    metric="cityblock",
)
fig_class_mixing_matrix.update_layout(height=600, width=600)
fig_class_mixing_matrix.show()

<a name="section-five"></a>
## 5. Pairwise space comparison

### 5.1 Global comparison of pairwise distances

The `plot_pairwise_distance_comparison` function generates a scatter plot to compare pairwise distances between samples in two different embedding spaces. Using a specified distance metric (default: euclidean), it shows the distances for the same set of samples across both embedding spaces.
Additionally it computes the Spearman correlation coefficient between the pairwise distances in the two selected embedding spaces.
The function allows customization of plot title, color, and scatter dot opacity, and optionally groups points based on a meta data feature, enabling insights into how different sample categories behave across embeddings.

In [ ]:
from emmaemb.visualization import plot_pairwise_distance_comparison

fig_pwd_comparison = plot_pairwise_distance_comparison(
    emma,
    emb_space_y="ProtT5",
    emb_space_x="ESMC",
    metric="cityblock",
    group_by="species",
)
fig_pwd_comparison.update_layout(height=600, width=600)
fig_pwd_comparison.show()

### 5.2 Cross-space neighborhood similarity

The `plot_low_similarity_distribution` function visualizes the class distribution of samples with low neighborhood similarity between two embedding spaces. It computes neighborhood similarity scores based on a specified distance metric (default: euclidean) and identifies samples where similarity of the nearest neighbors of a data point falls below a given threshold. The function then compares the class distribution of these low-similarity samples to the overall dataset distribution using a scatter plot. This helps assess whether certain classes exhibit higher or lower structural consistency across embeddings, providing insights into differences in how embeddings capture relationships between samples.

In [ ]:
from emmaemb.visualization import plot_low_similarity_distribution

fig_low_similarity_class_distribution = plot_low_similarity_distribution(
    emma,
    emb_space_1="ProtT5",
    emb_space_2="ESMC",
    feature="enzyme_class",
    k=10,
    metric="cityblock",
    similarity_threshold=0.3,
)
fig_low_similarity_class_distribution.update_layout(height=600, width=600)
fig_low_similarity_class_distribution.show()